# 07 — Clasificación del corpus español: BERTopic + Zero-Shot (BART-MNLI)

Clasificación del corpus español (`scam_es_FINAL_v2.csv`) con el **mismo
clasificador validado para el corpus estadounidense**: BART-MNLI con etiquetas
en inglés.

## Por qué BART y no mDeBERTa

Una primera iteración empleó el modelo multilingüe mDeBERTa-v3-XNLI con
etiquetas en español. La inspección de resultados reveló una clasificación
inconsistente: la categoría `charity` absorbió el 45% del corpus con confianza
media de 0,43, concentrando tweets procedentes en un 88,7% de consultas de
phishing. El modelo empleaba esa categoría como clase residual no fiable.

Se adopta por tanto el clasificador validado en el corpus EEUU
(`facebook/bart-large-mnli` + etiquetas en inglés). Esta decisión:
- Resuelve la inconsistencia detectada.
- Garantiza la **plena comparabilidad** entre el corpus EEUU y el español
  (mismo modelo, mismas etiquetas).

El notebook de mDeBERTa se conserva en `experimentos/` para trazabilidad.

## Antes de Run All

- ✅ Mac enchufado.
- ✅ `caffeinate -dimsu` en otra terminal.
- ✅ Kernel: **Python (tfm-nlp)**.
- ⏱️ Estimación: 3-5 horas (corpus de 1.229 tweets, CPU Intel).


## Verificación de entorno

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy, torch, transformers, sentence_transformers, bertopic, pandas
print(f"numpy:                 {numpy.__version__}")
print(f"torch:                 {torch.__version__}")
print(f"transformers:          {transformers.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"bertopic:              {bertopic.__version__}")
print(f"pandas:                {pandas.__version__}")
print()
print("✓ Entorno listo")


## Carga del corpus español

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import time

pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_es_FINAL_v2.csv")
print(f"Tweets cargados: {len(df)}")

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_analysis(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean_for_analysis)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    mask_empty = df["clean_text"] == ""
    df.loc[mask_empty, "clean_text"] = df.loc[mask_empty, "text"].apply(clean_for_analysis)

df = df[df["clean_text"].str.len() >= 20].reset_index(drop=True)
print(f"Tras filtrar textos cortos: {len(df)}")

docs = df["clean_text"].tolist()


## FASE 1 — BERTopic

⏳ ~5-10 minutos. `min_topic_size=20` por ser un corpus más pequeño.
El modelo de embeddings es multilingüe y procesa el texto español sin problema.


In [ ]:
print("⏳ Iniciando BERTopic (corpus español)...")
t0 = time.time()

from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    min_topic_size=20,
    nr_topics="auto",
    calculate_probabilities=False,
    language="multilingual",
    verbose=True,
)

print("⏳ Calculando embeddings y agrupando...")
topics, _ = topic_model.fit_transform(docs)

t_bertopic = time.time() - t0
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f"\n✓ BERTopic completado en {t_bertopic/60:.1f} min")
print(f"  Tópicos encontrados: {n_topics}")
print(f"  Tweets en outliers (-1): {n_outliers}")


In [ ]:
topic_info = topic_model.get_topic_info()
print("=== INFO DE TÓPICOS ===")
print(topic_info.head(25))


In [ ]:
df["bertopic_id"] = topics

topic_keywords = {}
for tid in set(topics):
    if tid == -1:
        topic_keywords[tid] = "outliers"
        continue
    words = topic_model.get_topic(tid)
    topic_keywords[tid] = ", ".join([w for w, _ in words[:8]]) if words else ""

df["bertopic_keywords"] = df["bertopic_id"].map(topic_keywords)

print("=== RESUMEN BERTOPIC ===\n")
for tid in sorted(set(topics)):
    n = (df["bertopic_id"] == tid).sum()
    kw = topic_keywords.get(tid, "")
    print(f"  Tópico {tid:>3} ({n:>4} tweets):  {kw[:80]}")


In [ ]:
print("=== EJEMPLOS POR TÓPICO ===\n")
shown = 0
for tid in sorted(set(topics)):
    if shown >= 12:
        break
    subset = df[df["bertopic_id"] == tid]
    kw = topic_keywords.get(tid, "")
    print(f"--- TÓPICO {tid} ({len(subset)} tweets) ---")
    print(f"    Keywords: {kw}")
    for _, row in subset.head(2).iterrows():
        print(f"    • {str(row['text'])[:200]}")
    print()
    shown += 1


## Guardado intermedio (post-BERTopic)

In [ ]:
df.to_csv("../data/raw/scam_es_with_bertopic.csv", index=False, encoding="utf-8")
print(f"✓ Guardado intermedio: scam_es_with_bertopic.csv ({len(df)} filas)")


## FASE 2 — Clasificación Zero-Shot con BART-MNLI

Las 14 categorías de la memoria, **en inglés** (idénticas a las del corpus EEUU).
El modelo BART entiende texto español por su capacidad multilingüe heredada;
mantener etiquetas en inglés asegura la comparabilidad con el corpus EEUU.

⏳ 3-5 horas en CPU Intel.


In [ ]:
CANDIDATE_LABELS = [
    "investment or cryptocurrency scam",
    "romance scam",
    "phishing or identity theft scam",
    "government agency impersonation scam",
    "bank fraud or unauthorized wire transfer",
    "payment app scam (Zelle, Venmo, Bizum, PayPal)",
    "Ponzi or pyramid scheme",
    "tech support scam",
    "employment or job offer scam",
    "charity or donation scam",
    "insurance fraud",
    "corporate or securities fraud",
    "tax fraud or tax scam",
    "not related to financial fraud",
]

LABEL_TO_CODE = {
    "investment or cryptocurrency scam": "investment_crypto",
    "romance scam": "romance",
    "phishing or identity theft scam": "phishing_identity",
    "government agency impersonation scam": "gov_impersonation",
    "bank fraud or unauthorized wire transfer": "bank_wire",
    "payment app scam (Zelle, Venmo, Bizum, PayPal)": "payment_app",
    "Ponzi or pyramid scheme": "ponzi_pyramid",
    "tech support scam": "tech_support",
    "employment or job offer scam": "employment",
    "charity or donation scam": "charity",
    "insurance fraud": "insurance",
    "corporate or securities fraud": "corporate",
    "tax fraud or tax scam": "tax",
    "not related to financial fraud": "not_related",
}

print(f"Categorías candidatas (inglés, idénticas a EEUU): {len(CANDIDATE_LABELS)}\n")
for i, lbl in enumerate(CANDIDATE_LABELS, 1):
    print(f"  {i:2d}. {lbl}")


In [ ]:
print("⏳ Cargando modelo BART-MNLI...")
t0 = time.time()

from transformers import pipeline
import torch

device = -1  # CPU
print(f"   Dispositivo: CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

t_load = time.time() - t0
print(f"✓ Modelo cargado en {t_load/60:.1f} min")


In [ ]:
texts_to_classify = df["clean_text"].tolist()
n = len(texts_to_classify)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 50

predictions = []
scores = []

print(f"⏳ Clasificando {n} tweets en {len(CANDIDATE_LABELS)} categorías (BART)...")
print(f"   Lote: {BATCH_SIZE} | Checkpoint cada {CHECKPOINT_EVERY}")
print(f"   ESTIMACIÓN: 3-5 horas en CPU Intel.\n")

t0 = time.time()
last_print = t0

for i in range(0, n, BATCH_SIZE):
    batch = texts_to_classify[i:i+BATCH_SIZE]

    try:
        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]
    except Exception as e:
        print(f"  ⚠️ Error en lote {i}: {e}")
        for _ in batch:
            predictions.append("not related to financial fraud")
            scores.append(0.0)
        continue

    for r in results:
        predictions.append(r["labels"][0])
        scores.append(r["scores"][0])

    if (time.time() - last_print) > 30 or (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        done = min(i + BATCH_SIZE, n)
        eta = (elapsed / done) * (n - done) if done > 0 else 0
        print(f"  {done}/{n} ({done/n*100:.1f}%) — {elapsed/60:.1f} min — ETA: {eta/60:.1f} min")
        last_print = time.time()

    if (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0 and (i + BATCH_SIZE) < n:
        df_partial = df.iloc[:len(predictions)].copy()
        df_partial["predicted_label"] = predictions
        df_partial["confidence_score"] = scores
        df_partial.to_csv("../data/raw/scam_es_classification_bart_checkpoint.csv", index=False)

t_classify = time.time() - t0
print(f"\n✓ Clasificación completada en {t_classify/60:.1f} min")


In [ ]:
df["predicted_label"] = predictions[:len(df)]
df["confidence_score"] = scores[:len(df)]
df["predicted_category"] = df["predicted_label"].map(LABEL_TO_CODE)
df["is_relevant"] = df["predicted_category"] != "not_related"

print("=== DISTRIBUCIÓN DE CATEGORÍAS PREDICHAS (ESPAÑA - BART) ===\n")
counts = df["predicted_category"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes: {df['is_relevant'].sum()} / {total} ({df['is_relevant'].mean()*100:.1f}%)")


In [ ]:
print("=== CONFIANZA (ESPAÑA - BART) ===\n")
print(f"  Media:    {df['confidence_score'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score'].median():.3f}")
print(f"  Min:      {df['confidence_score'].min():.3f}")
print(f"  Max:      {df['confidence_score'].max():.3f}")
print(f"\n  Alta confianza (>=0.7):  {(df['confidence_score'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):         {((df['confidence_score'] >= 0.4) & (df['confidence_score'] < 0.7)).sum()}")
print(f"  Baja (<0.4):             {(df['confidence_score'] < 0.4).sum()}")


## Verificación rápida — ¿charity sigue inflada?

Control de calidad: charity NO debería volver a absorber el corpus.


In [ ]:
charity_n = (df["predicted_category"] == "charity").sum()
phish_n = (df["predicted_category"] == "phishing_identity").sum()
print(f"charity:            {charity_n} ({charity_n/len(df)*100:.1f}%)")
print(f"phishing_identity:  {phish_n} ({phish_n/len(df)*100:.1f}%)")
print()
if charity_n > len(df) * 0.25:
    print("⚠️ ATENCIÓN: charity sigue alta. Revisar ejemplos antes de dar por bueno.")
else:
    print("✓ charity en rango razonable.")
print()
print("Recordatorio: la query 'phishing' era ~74% del corpus crudo,")
print("por lo que phishing_identity debería ser una categoría destacada.")


## Guardado final

In [ ]:
OUTPUT = "../data/raw/scam_es_FINAL_classified.csv"
df.to_csv(OUTPUT, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT}")
print(f"  Total filas: {len(df)}")
print()
print("📦 Este CSV sustituye al de mDeBERTa (clasificación definitiva con BART).")
print()
print("🚨 HAZ COPIA DE SEGURIDAD en Drive.")
print()
print("📌 SIGUIENTE: análisis comparativo EEUU vs España.")


## Inspección de resultados

In [ ]:
print("=== 3 EJEMPLOS DE CADA CATEGORÍA (España - BART, alta confianza) ===\n")
for cat_code in df["predicted_category"].value_counts().index:
    subset = df[
        (df["predicted_category"] == cat_code) &
        (df["confidence_score"] >= 0.5)
    ].nlargest(3, "confidence_score")
    n_total = (df["predicted_category"] == cat_code).sum()
    print(f"--- {cat_code} ({n_total} total) ---")
    for _, row in subset.iterrows():
        print(f"  [{row['confidence_score']:.2f}] {str(row['text'])[:200]}")
    print()
